In [ ]:
import os
import json
from dotenv import load_dotenv
from pydantic import (BaseModel, Field)
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)
from azure.storage.blob import BlobServiceClient
from langchain_openai import (ChatOpenAI, OpenAIEmbeddings)
from langgraph.graph import (StateGraph, START, END)
from langchain.agents import create_agent

In [ ]:
load_dotenv()

In [ ]:
azure_project_endpoint = os.getenv("AZURE_PROJECT_ENDPOINT")
azure_model_endpoint = os.getenv("AZURE_MODEL_ENDPOINT")
azure_model_name = os.getenv("AZURE_MODEL_NAME")
azure_embedding_model_name = os.getenv("AZURE_EMBEDDING_MODEL_NAME")

In [ ]:
credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [ ]:
llm = ChatOpenAI(
    model=azure_model_name,
    base_url=f"{azure_project_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
    use_responses_api=True,
)

In [ ]:
def load_employee_data():
    """Download employee JSON data from Azure Blob Storage."""
    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_EMPLOYEES"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_EMPLOYEES"]

    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"employees": [...]}.
        if isinstance(data, dict):
            data = data.get("employees")

        if not isinstance(data, list):
            raise ValueError(
                "Employee data must be a JSON list or an "
                "object containing an 'employees' list."
            )

        return data

    finally:
        credential.close()

In [ ]:
def load_task_data():
    """Download task JSON data from Azure Blob Storage."""

    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_TASKS"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_TASKS"]
    
    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"tasks": [...]}.
        if isinstance(data, dict):
            data = data.get("tasks")

        if not isinstance(data, list):
            raise ValueError(
                "Task data must be a JSON list or an "
                "object containing a 'tasks' list."
            )

        return data

    finally:
        credential.close()

In [ ]:
employee_data = load_employee_data()
print('employee_data ', employee_data)

In [ ]:
task_data = load_task_data()
print('task_data ', task_data)

In [ ]:
def evaluate_employee_skill_set(employee_name, skill, level, evalation_agent):
    """
    This function returns an evaluation of a particular employee's skills.
    """

    evaluation_prompt = f"""
        The employee's name is {employee_name}.
        The skill under evaluation is {skill}.
        The level of expertise is {level}.
    """

    result = evaluation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    })
    return result

In [ ]:
evaluation_agent_system_prompt = (
    "You are an employee skill evaluator. "
    "Your job is to read an employee's skill as well as how advanced they are at that particular skill. "
    "You must output a brief statement about their level at that skill."
)

evaluation_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt
)

In [ ]:
evaluations = []
for employee in employee_data:
    for skill, level in employee['skill_map'].items():
        evaluation = evaluate_employee_skill_set(employee['name'], skill, level, evaluation_agent)
        evaluations.append(evaluation)

In [ ]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(
    model=azure_embedding_model_name,
    base_url=f"{azure_model_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

In [ ]:
evaluation_content = []
for evaluation in evaluations:
    content = evaluation['messages'][1].content
    evaluation_content.append(content[0]['text'])

In [ ]:
task_data

In [ ]:
evaluation_content

In [ ]:
enum 

class TaskEmployeeCandidate(BaseModel):
    #task_id: int
    #employee_id: int
    learning_opportunity_score: float = Field(
        ge=0,
        le=10,
        description="Employee's learning opportunity score."
    )
    leadership_capability_score: float = Field(
        ge=0,
        le=10,
        description="Employee's leadership capability score."
    )
    learning_explanation: str
    leadership_explanation: str

In [ ]:
skill_matching_agent_system_prompt = (
    "You are a technical skill evaluation and employee matching agent. "

    "YOUR OBJECTIVE: "
    "Evaluate an employee's technical skills and proficiency using "
    "a list of required skills and the available employee evaluation. "
    "Your goal is to identify both the employee's potential for learning "
    "and their ability to provide technical leadership to others. "

    "SKILL EVALUATION: "
    "Evaluate each listed skill individually. "
    "Use the provided evaluation as the primary source of evidence. "
    "You may infer transferable knowledge when the employee has experience "
    "with related technologies or concepts. For example, Angular experience "
    "may indicate familiarity with CSS because Angular applications commonly "
    "use CSS for styling. However, related experience does not automatically "
    "prove proficiency in the associated skill. "
    "Distinguish between demonstrated proficiency and inferred proficiency. "
    "Never invent experience or assume expertise without supporting evidence. "

    "If a listed skill is not mentioned in the evaluation and there is "
    "insufficient evidence to infer proficiency, treat that skill as unknown. "
    "Do not automatically classify unknown skills

skill_matching_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt,
    response_format=TaskEmployeeCandidate
)

In [ ]:
def match_task_to_skills(skills, evaluation, skill_matching_agent):
    skill_matching_prompt = f"""
        The list of skills are {skills}.
        The evaluation is {evaluation}.
    """
    result = skill_matching_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": skill_matching_prompt
            }
        ]
    })
    return result

In [ ]:
skill_matches = []
for task in task_data:
    skills = ", ".join(task['required_skills'])
    for evaluation in evaluation_content:
        skill_match = match_task_to_skills(skills, evaluation, skill_matching_agent)
        skill_matches.append(skill_match)

In [ ]:
for skill_match in skill_matches:
    print(skill_match)
    print('-')